# 🛰️ Agronavis — ISRO PS-6 Hackathon Pipeline

**AI-Driven Automated Crop Type, Moisture Stress Detection & Irrigation Advisory**

---

## ✅ How to Use This Notebook

**Only one cell needs to be edited before running:** the `⚙️ CONFIGURATION` cell below.

1. Set your `AOI_COORDS` polygon (paste your own coordinates).
2. Set your `GEE_PROJECT_ID`.
3. Set your date range (`START_DATE`, `END_DATE`).
4. **Run All Cells** from top to bottom.

Everything else is fully automated.

---

### Pipeline Stages
| Step | What happens |
|------|--------------|
| **0** | Setup & Authentication |
| **1** | Export Sentinel-2 (Optical) & Sentinel-1 (SAR) composites to Drive |
| **2** | Export ERA5 daily weather data to Drive as CSV |
| **3** | Generate automated ground-truth points (ESA WorldCover) |
| **4** | Feature extraction (NDVI, NDWI, SAR backscatter) from downloaded TIFs |
| **5** | Train Random Forest classifier & evaluate (Accuracy, Kappa) |
| **6** | Generate full-scene classified crop map (GeoTIFF) |

In [ ]:
# ============================================================
# ⚙️  CONFIGURATION — EDIT THIS CELL ONLY
# ============================================================

# --- 1. Your Area of Interest (AOI) ---
# Paste your polygon ring coordinates here as [longitude, latitude] pairs.
# The first and last point MUST be identical to close the ring.
AOI_COORDS = [
    [84.1922093, 25.603894],
    [84.1939225, 25.6025315],
    [84.1969758, 25.6050228],
    [84.1940965, 25.6068942],
    [84.1922093, 25.603894]   # Close the ring
]

# --- 2. Google Earth Engine Project ID ---
GEE_PROJECT_ID = 'agri-recommendation-engine'  # Replace with your GEE project ID

# --- 3. Date range for satellite composites ---
START_DATE = '2025-08-01'
END_DATE   = '2025-10-31'

# --- 4. Google Drive folder for all outputs ---
DRIVE_FOLDER = 'Agronavis_Hackathon_Data'

# --- 5. Cloud cover threshold for Sentinel-2 ---
MAX_CLOUD_PCT = 20

# --- 6. Buffer (metres) around AOI for ground-truth sampling ---
GT_BUFFER_M = 2000

# --- 7. Number of ground-truth points per class ---
GT_CROP_POINTS    = 50
GT_NONCROP_POINTS = 30

print('✅ Configuration loaded.')
print(f'   AOI has {len(AOI_COORDS)} vertices')
print(f'   Date range: {START_DATE} → {END_DATE}')
print(f'   Drive folder: {DRIVE_FOLDER}')

---
## Step 0 — Setup & Authentication

In [ ]:
# ============================================================
# Step 0: Mount Drive & Authenticate GEE
# ============================================================
import ee
import google.colab

# Mount Google Drive so all outputs are saved persistently
google.colab.drive.mount('/content/drive')

# Authenticate and initialise Earth Engine
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT_ID)

# ---- Derived GEE objects (shared across all GEE cells) ------
aoi         = ee.Geometry.Polygon([AOI_COORDS])
export_region = aoi.buffer(GT_BUFFER_M)   # Larger region for TIF exports

# ---- Derived local paths (shared across local processing cells) ----
DRIVE_BASE      = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
OPTICAL_TIF     = f'{DRIVE_BASE}/S2_Optical_Composite_AOI.tif'
SAR_TIF         = f'{DRIVE_BASE}/S1_SAR_Composite_AOI.tif'
GROUND_TRUTH    = f'{DRIVE_BASE}/Automated_Ground_Truth.geojson'
TRAINING_CSV    = f'{DRIVE_BASE}/ml_training_data.csv'
CLASSIFIED_TIF  = f'{DRIVE_BASE}/Classified_Crop_Map.tif'

print('✅ GEE initialised.')
print(f'   All outputs will be written to: {DRIVE_BASE}')

---
## Step 1 — Export Sentinel-2 (Optical) & Sentinel-1 (SAR) Composites

In [ ]:
# ============================================================
# Step 1: Export Sentinel-2 & Sentinel-1 median composites
# ============================================================

# --- Sentinel-2 (Optical) ---
# Bands: B3=Green, B4=Red, B8=NIR, B11=SWIR
s2_composite = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(export_region)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PCT))
    .select(['B3', 'B4', 'B8', 'B11'])
    .median()
    .clip(export_region)
)

task_s2 = ee.batch.Export.image.toDrive(
    image=s2_composite,
    description='S2_Optical_Composite_AOI',
    folder=DRIVE_FOLDER,
    scale=10,
    region=export_region,
    maxPixels=1e8
)
task_s2.start()
print('📤 Sentinel-2 export task started → check GEE Tasks tab.')

# --- Sentinel-1 (SAR) ---
# Bands: VV, VH
s1_composite = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(export_region)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select(['VV', 'VH'])
    .median()
    .clip(export_region)
)

task_s1 = ee.batch.Export.image.toDrive(
    image=s1_composite,
    description='S1_SAR_Composite_AOI',
    folder=DRIVE_FOLDER,
    scale=10,
    region=export_region,
    maxPixels=1e8
)
task_s1.start()
print('📤 Sentinel-1 export task started → check GEE Tasks tab.')
print()
print('⏳ Wait until BOTH tasks complete in your GEE Tasks panel before proceeding.')

---
## Step 2 — Export ERA5 Daily Weather Data

In [ ]:
# ============================================================
# Step 2: Export ERA5-Land daily weather to Google Drive CSV
# ============================================================
print('⛅ Fetching daily ERA5 weather data ...')

era5 = (
    ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .select(['temperature_2m', 'total_precipitation_sum', 'potential_evaporation_sum'])
)

def extract_weather(image):
    date  = image.date().format('YYYY-MM-DD')
    stats = image.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=aoi.centroid(),
        scale=1000,
        bestEffort=True
    )
    return ee.Feature(None, {
        'Date':             date,
        'Temp_2m_Kelvin':   stats.get('temperature_2m'),
        'Precipitation_m':  stats.get('total_precipitation_sum'),
        'Potential_Evap_m': stats.get('potential_evaporation_sum')
    })

weather_fc = era5.map(extract_weather)

task_weather = ee.batch.Export.table.toDrive(
    collection=weather_fc,
    description='ERA5_Weather_Data_AOI',
    folder=DRIVE_FOLDER,
    fileFormat='CSV'
)
task_weather.start()
print('📤 ERA5 weather CSV export task started → check GEE Tasks tab.')

---
## Step 3 — Generate Automated Ground-Truth Points (ESA WorldCover)

In [ ]:
# ============================================================
# Step 3: Generate labelled ground-truth points via WorldCover
# ESA WorldCover classes used:
#   40 = Cropland   → label 1
#   50 = Built-up  }
#   80 = Water      } → label 0
# ============================================================
print('📍 Generating smart ground-truth points ...')

sampling_region = aoi.buffer(GT_BUFFER_M)
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().clip(sampling_region)

crop_mask     = worldcover.eq(40)
non_crop_mask = worldcover.eq(50).Or(worldcover.eq(80))

crop_points = (
    ee.FeatureCollection.randomPoints(region=sampling_region, points=GT_CROP_POINTS, seed=42)
    .map(lambda f: f.set('class', 1))
)
non_crop_points = (
    ee.FeatureCollection.randomPoints(region=sampling_region, points=GT_NONCROP_POINTS, seed=101)
    .map(lambda f: f.set('class', 0))
)

ground_truth_fc = crop_points.merge(non_crop_points)

task_gt = ee.batch.Export.table.toDrive(
    collection=ground_truth_fc,
    description='Automated_Ground_Truth',
    folder=DRIVE_FOLDER,
    fileFormat='GeoJSON'
)
task_gt.start()
print('📤 Ground-truth GeoJSON export task started → check GEE Tasks tab.')
print()
print('⏳ Wait until ALL three tasks (Steps 1-3) complete before running Step 4.')

---
## Step 4 — Feature Extraction from Downloaded TIFs

> **Pre-requisite:** Steps 1 & 3 must have finished and the TIF files must be in your Drive.

In [ ]:
# ============================================================
# Step 4: Extract spectral features from TIF files
# Outputs: ml_training_data.csv
# ============================================================
import rasterio
import geopandas as gpd
import pandas as pd
import numpy as np

print('📂 Loading ground-truth points ...')
points = gpd.read_file(GROUND_TRUTH)
print(f'   Loaded {len(points)} points.')

# ---- Optical features (Sentinel-2) ----
print('🌿 Extracting Sentinel-2 (Optical) features ...')
with rasterio.open(OPTICAL_TIF) as src:
    points_opt = points.to_crs(src.crs)
    coords_opt = list(zip(points_opt.geometry.x, points_opt.geometry.y))
    optical_values = list(src.sample(coords_opt))

optical_df = pd.DataFrame(optical_values, columns=['Green', 'Red', 'NIR', 'SWIR'])
optical_df['NDVI'] = (optical_df['NIR'] - optical_df['Red']) / (optical_df['NIR'] + optical_df['Red'] + 1e-8)
optical_df['NDWI'] = (optical_df['Green'] - optical_df['NIR']) / (optical_df['Green'] + optical_df['NIR'] + 1e-8)

# ---- SAR features (Sentinel-1) ----
print('📡 Extracting Sentinel-1 (SAR) features ...')
try:
    with rasterio.open(SAR_TIF) as src:
        points_sar = points.to_crs(src.crs)
        coords_sar = list(zip(points_sar.geometry.x, points_sar.geometry.y))
        sar_values = list(src.sample(coords_sar))
    sar_df = pd.DataFrame(sar_values, columns=['VV', 'VH'])
    sar_df['VH_VV_Ratio'] = sar_df['VH'] / (sar_df['VV'] + 1e-8)
    sar_available = True
except Exception as e:
    print(f'   ⚠️  SAR file not available ({e}). Proceeding with optical-only features.')
    sar_df = pd.DataFrame(0.0, index=np.arange(len(optical_df)), columns=['VV', 'VH', 'VH_VV_Ratio'])
    sar_available = False

# ---- Combine ----
print('🔗 Building feature stack ...')
final_df = pd.concat([optical_df, sar_df, points[['class']].reset_index(drop=True)], axis=1)

# Remove rows where optical data is completely missing (edge-of-image zeros)
final_df = final_df[final_df['NIR'] != 0].dropna()

# Choose feature set based on SAR availability
if sar_available and final_df['VV'].mean() != 0:
    FEATURE_COLS = ['NDVI', 'NDWI', 'Green', 'Red', 'NIR', 'SWIR', 'VV', 'VH', 'VH_VV_Ratio']
    print('   ✅ SAR data valid — using full optical + SAR feature set.')
else:
    FEATURE_COLS = ['NDVI', 'NDWI', 'Green', 'Red', 'NIR', 'SWIR']
    print('   ℹ️  SAR data empty — using optical-only feature set.')

final_df = final_df[FEATURE_COLS + ['class']]
final_df.to_csv(TRAINING_CSV, index=False)

print(f'\n✅ Saved {len(final_df)} valid rows → {TRAINING_CSV}')
print(f'   Feature columns: {FEATURE_COLS}')
print('\n--- Dataset Preview ---')
print(final_df.head())
print(f'\n   Class distribution:\n{final_df["class"].value_counts().to_string()}')

---
## Step 5 — Train Random Forest & Evaluate

In [ ]:
# ============================================================
# Step 5: Train Random Forest classifier & evaluate
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report, confusion_matrix

print('📊 Loading training data ...')
df = pd.read_csv(TRAINING_CSV)

FEATURE_COLS = [c for c in df.columns if c != 'class']
X = df[FEATURE_COLS]
y = df['class']

# 70/30 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f'   Training samples : {len(X_train)}')
print(f'   Test samples     : {len(X_test)}')
print(f'   Features used    : {FEATURE_COLS}')

# --- Train ---
print('\n🌲 Training Random Forest ...')
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# --- Evaluate ---
y_pred   = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
kappa    = cohen_kappa_score(y_test, y_pred)

print()
print('=' * 45)
print('🎯  ISRO PS-6 — EVALUATION METRICS')
print('=' * 45)
print(f'   Overall Accuracy : {accuracy * 100:.2f}%  (Target: ≥85%)')
print(f'   Kappa Coefficient: {kappa:.4f}')
print('=' * 45)
print()
print('Detailed Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Non-Crop (0)', 'Crop (1)']))

# --- Confusion matrix ---
plt.figure(figsize=(6, 4))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Crop', 'Crop'],
            yticklabels=['Non-Crop', 'Crop'])
plt.title('Crop Classification — Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# --- Feature importance ---
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
importances.plot(kind='bar', color='steelblue')
plt.title('Random Forest — Feature Importances')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

---
## Step 6 — Generate Full-Scene Classified Crop Map (GeoTIFF)

In [ ]:
# ============================================================
# Step 6: Apply trained model to every pixel → classified map
# ============================================================
import rasterio
import numpy as np

print('🗺️  Applying model to generate full spatial crop map ...')

with rasterio.open(OPTICAL_TIF) as src:
    meta = src.meta.copy()

    green = src.read(1).astype(float)
    red   = src.read(2).astype(float)
    nir   = src.read(3).astype(float)
    swir  = src.read(4).astype(float)

    np.seterr(divide='ignore', invalid='ignore')
    ndvi = np.where((nir + red)   == 0., 0., (nir - red)   / (nir + red))
    ndwi = np.where((green + nir) == 0., 0., (green - nir) / (green + nir))

    # Build the same feature stack as training
    bands_to_stack = [ndvi, ndwi, green, red, nir, swir]

    # Optionally add SAR if it was used in training
    if 'VV' in FEATURE_COLS:
        print('   Adding SAR bands from S1 TIF ...')
        with rasterio.open(SAR_TIF) as sar_src:
            vv = sar_src.read(1).astype(float)
            vh = sar_src.read(2).astype(float)
        vh_vv = vh / (vv + 1e-8)
        bands_to_stack += [vv, vh, vh_vv]

    img_stack = np.stack(bands_to_stack)          # (n_bands, H, W)
    n_bands, height, width = img_stack.shape

    flat_img = img_stack.reshape(n_bands, -1).T   # (H*W, n_bands)
    flat_img = np.nan_to_num(flat_img)

    print('   Predicting crop class for every pixel ...')
    predictions_flat = rf_model.predict(flat_img)
    classified_img   = predictions_flat.reshape(height, width)

    meta.update(count=1, dtype='uint8', nodata=255)

    print(f'   Saving classified map → {CLASSIFIED_TIF}')
    with rasterio.open(CLASSIFIED_TIF, 'w', **meta) as dst:
        dst.write(classified_img.astype('uint8'), 1)

print()
print('✅ Classified crop map saved!')
print(f'   File: {CLASSIFIED_TIF}')
print(f'   Size: {width} × {height} pixels')
unique, counts = np.unique(classified_img, return_counts=True)
for cls, cnt in zip(unique, counts):
    label = 'Crop' if cls == 1 else 'Non-Crop'
    pct   = cnt / (width * height) * 100
    print(f'   Class {cls} ({label}): {cnt:,} pixels ({pct:.1f}%)')